# OpenAI Integration
* In this notebook we'll integrate OpenAI and create a POC `ChatBot` that will answer our questions related to the show "The Office"

## Import Libraries

In [ ]:
from pathlib import Path
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import CharacterTextSplitter
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain.schema import Document
from langchain_openai import OpenAIEmbeddings


import os
import pandas as pd
import numpy as np
import chromadb


## Initialize DB

In [6]:
embedding_function = OpenAIEmbeddings()
vector_db = Chroma(persist_directory="../db", embedding_function=embedding_function)

In [7]:
## lets verify the db connection
vector_db._collection.count()

14998

In [8]:
## Retrieve using similarity search
query = "When did Michael say 'I love you'"
results = vector_db.similarity_search(query=query, k=4)

for doc in results:
    print(doc.page_content)
    print(doc.metadata)

in love on Valentine's Day. Holly: Two people in love? Michael: I love you. Holly: Wait, wait, wait, what do you mean you love me? We've only been dating for a week. Do you mean you love me like, 'oh, hey, there's Holly. I love that girl.' Or you do you mean you love me like you love me-love me? Michael: I love you-love you.
{'directed_by': 'Greg Daniels', 'episode': 15, 'episode_description': "It's Valentine's Day, and the office is fed up with Michael and Holly's PDA, Andy helps Erin solve Gabe's riddles to find her gift, and Jim and Pam get drunk and try to find a place in the office to have sex.", 'id': 's7_e15_scene23', 'rating': 8.4, 'scene': 23, 'season': 7, 'speakers': 'Holly,Michael', 'written_by': 'Robert Padnick'}
I'm too shy to tell you that I love you.' Michael: Pam.  Pam, you gave me your word. Ryan: [kissing Kelly against her desk] You did that for me? Kelly: Mmhmm. Ryan: Are you happy you did? Toby: Hey guys that's really inappropriate. Ryan: [kisses for a little longer

## Initialize LLMs

In [9]:
from langchain.chat_models import init_chat_model

model = init_chat_model("gpt-4.1-mini", model_provider="openai")

## Helper Functions

* Lets create following helper functions that will help us convert user query to openAI bot question
    * retrieve_documents - a function that takes in user query and number of results to return as params and returns the result
    * format_prompt - a function that takes in user question and context docs to create a prompt for the LLM
    * generate_response - make the OpenAI call using system and user prompt to get the response. 
    * ask_dunderbot - a wrapper function that takes in user query and returns the response using the above questions.

In [16]:
from langchain_core.prompts import ChatPromptTemplate

# helper function to query ChromaDB
def retrieve_documents(user_query, number_of_results=10):    
    results = vector_db.similarity_search(query=user_query, k=number_of_results)
    return results

system_template = """
    You are DunderBot 🤖 trained on quotes from The Office TV show.
    Use the following episode content to answer the user's question. Be fun, but don't make things up.
    ---------------------
    {context}
    ---------------------"""

prompt_template = ChatPromptTemplate.from_messages([
    ("system",system_template),("user","{user_query}")
])

# helper function to create prompt based on user_query and document context
def format_prompt(user_query, context_documents):
    context = "\n\n".join([f"Lines : {doc.page_content}\nInfo : {doc.metadata}" for doc in context_documents])
    return prompt_template.invoke({
        "context":context,
        "user_query":user_query
    })
    
    
def generate_response(prompt):
    reponse = model.invoke(prompt)
    return reponse.content

def ask_dunderbot(user_query, number_of_results=10):
    context_documents = retrieve_documents(user_query=user_query, number_of_results=number_of_results)
    prompt = format_prompt(user_query=user_query, context_documents=context_documents)
    answer = generate_response(prompt=prompt)
    return answer
    

In [17]:
question = "What did Michael say about his management style?"
answer = ask_dunderbot(question)
print("🤖 DunderBot says:\n")
print(answer)

🤖 DunderBot says:

Michael said his management style is that he "touches people's hearts and souls with humor, with love and maybe a dash of razzle-dazzle." So basically, he leads with a big ol' mix of heart and razzle-dazzle flair! Classic Michael.


In [18]:
question = "Can you find out how many episodes had discussion regarding bears"
answer = ask_dunderbot(question)
print("🤖 DunderBot says:\n")
print(answer)

🤖 DunderBot says:

Sure thing! From the quotes and episode details I found, discussions about bears appeared in these episodes:

1. Season 3, Episode 20 - Jim dresses as Dwight and they have the iconic "What kind of bear is best?" conversation.
2. Season 3, Episode 15 - Dwight talks about watching "Grizzly Man" and bear attacks.
3. Season 6, Episode 18 - Dwight compares their trash mess to beavers and other animals.
4. Season 7, Episode 11 - Multiple mentions of bear-related conversations during the Christmas party.
5. Season 7, Episode 21 - Michael and Dwight talk about black bears and Michael being a "salami" to a bear.
6. Season 8, Episode 11 - Bears come up during the bar trivia competition.
7. Season 9, Episode 23 - Andy receives a "bear hug," referencing a "mama grizzly."

So, that's 7 episodes with bear-related discussions or jokes. Bears definitely have a strong presence in The Office! 🐻


In [19]:
question = "Can you list the seasons and spisodes where someone said 'thats what she said'"
answer = ask_dunderbot(question, number_of_results=100)
print("🤖 DunderBot says:\n")
print(answer)

🤖 DunderBot says:

Absolutely, here are some seasons and episodes from The Office where someone said the iconic line "That's what she said":

- Season 2, Episode 2 ("Sexual Harassment")  
- Season 3, Episode 7 ("Branch Closing")  
- Season 3, Episode 17 ("Business School")  
- Season 4, Episode 7 ("Survivor Man")  
- Season 4, Episode 8 ("Dinner Party")  
- Season 4, Episode 9 ("The Client")  
- Season 6, Episode 18 ("The Delivery")  
- Season 7, Episode 4 ("Sex Ed")  
- Season 7, Episode 20 ("Training Day")  
- Season 9, Episode 23 ("Finale")  

Michael and Jim are the usual suspects who drop this hilarious line, often to lighten the mood or drive a joke. Want some classic "That's what she said" moments? Just ask!


In [20]:
question = "I Want some classic \"That's what she said\" moments"
answer = ask_dunderbot(question, number_of_results=100)
print("🤖 DunderBot says:\n")
print(answer)

🤖 DunderBot says:

Oh, classic "That's what she said" moments? I've got a few gems from Michael and the gang that will have you laughing like you're right there in the Scranton office!

1. Michael: "I never know. I just say it. I say stuff like that, you know, to lighten the tension. When things sort of get hard."
Jim: "That's what she said."
Michael: "Hey! Nice. Really good. Bravo, my young ward."  
(Source: s4_e7_scene51)

2. Jim: "Uh huh. Yeah, just wait. Ten years, you'll figure it out."
Michael: "That's what I said. That's what she said."
Jim: "That's what who said?"  
(Source: s4_e7_scene50)

3. Michael: "All day. Our favorite names, silly made up names, normal names said in a silly voice. Wouldn't that be nice?"
Andy: "I would like that."  
And then, classic:  
Michael: "That's what she said."  
(Source: s8_e6_scene5 and others)

4. Dwight: "All right!"
Michael: "I want you to get your ass out of my face."
Jim: "[sitting on a stack of paper] Yeah, well, if you're only free till 

In [21]:
question = "Which episode did Holly and Michael get engaged?"
answer = ask_dunderbot(question, number_of_results=100)
print("🤖 DunderBot says:\n")
print(answer)

🤖 DunderBot says:

Holly and Michael got engaged in Season 7, Episode 18. In that episode, Michael plans to propose to Holly during the Dunder Mifflin garage sale. Despite some obstacles like Holly possibly needing to take an extended leave of absence to Colorado because her father is sick, the engagement happens. The scenes show Michael's heartfelt proposal with lit candles and some humorous interaction with other coworkers also asking Holly to marry them (all receiving a "No"). The episode ends with Michael and Holly deciding to move to Colorado together.

So, the engagement episode is Season 7, Episode 18!


In [22]:
question = "How much did Michael pay for his condo?"
answer = ask_dunderbot(question, number_of_results=100)
print("🤖 DunderBot says:\n")
print(answer)

🤖 DunderBot says:

Michael paid $400 for his condo. As he said, "Oh! I don't know, Pam. I paid $400 for this phone because I liked the ring." (Note: This is a joke about a phone, but the actual condo price isn't directly stated. However, in the context of the condo, Michael mentioned selling it on eBay at 80% of what he paid, implying the price was more realistic.)

But to be more precise about his condo purchase, in the episode where he's finalizing his condo deal, he talks about it being a three-bedroom, two-bath contemporary townhouse with two-car parking and wall-to-wall carpets, but the exact price he paid isn't explicitly mentioned.

So, the exact purchase price of Michael's condo isn't clearly given in these quotes, but from the passage where he says he sold it on eBay "for eighty percent of what I paid," we can infer he paid more than the sale price - the sale price isn't quoted either. Sorry to leave you hanging, but Michael's condo price remains a bit of a mystery!
